In [23]:
# Tableau용 + 분석용 최종 전처리

In [24]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 공통 점검 함수

In [25]:
# 컬럼 정보 간단 표현
def check_basic_info(df, df_name, exclude_cols=None):
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")


    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

    print("[테이블 요약]")
    display(df.head())

In [26]:
# 범주형 컬럼 분포 확인 함수
def check_category_summary(df, df_name, col_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")
    
    # 컬럼이 실제로 존재하는지 먼저 확인
    df_copied = df.copy()           # 원본 데이터 훼손 방지
    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return
    
    # 범주별 개수를 계산 (결측치 포함 집계)
    summary_df = df_copied[col_name].value_counts(dropna=False).reset_index()
    
    # 결과 컬럼명 정리
    summary_df.columns = [col_name, '개수']
    
    # 전체 행 수 대비 각 범주의 비율(%) 계산
    summary_df['비율(%)'] = (summary_df['개수'] / len(df_copied) * 100).round(2)
    
    # 기본 요약 정보 출력
    print("전체 행 수:", len(df_copied))
    print(f"{col_name} 고유값 개수(결측 포함):", df_copied[col_name].nunique(dropna=False))
    print()
    
    # 범주별 요약표 상위 10개 출력
    display(summary_df.head(10))

# 데이터 로드 및 원본 구조 확인

In [27]:
# 데이터 로드
df = pd.read_csv("data/transcript_portfolio_profile.csv")

# 원본 훼손 방지를 위해 복제본 사용
df2 = df.copy()
check_basic_info(df2, 'transcript_portfolio_profile')


transcript_portfolio_profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,20
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
offer_reward,float64,167184,54.61,138953,45.39,5
difficulty,float64,167184,54.61,138953,45.39,5
duration,float64,167184,54.61,138953,45.39,5
channels,str,167184,54.61,138953,45.39,4
offer_type,str,167184,54.61,138953,45.39,3
web,float64,167184,54.61,138953,45.39,2
mobile,float64,167184,54.61,138953,45.39,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,has_profile
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,1
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0


In [28]:
# ============================================================
# 이벤트별 구조적 결측치 확인
# event 종류에 따라 원래 비어 있어야 하는 컬럼이 다르므로
# offer_id, amount, event_reward의 결측 패턴을 event별로 확인
# ============================================================
struct_missing_summary = (
    df2.groupby('event')[['offer_id', 'amount', 'event_reward']]
    .agg(lambda s: s.isna().sum())
)

display(struct_missing_summary)

,offer_id,amount,event_reward
event,,,
offer completed,0,33182,0
offer received,0,76277,76277
offer viewed,0,57725,57725
transaction,138953,0,138953


## 결측 해석 주의

데이터의 일부 결측은 데이터 오류가 아니라 이벤트 구조 차이에서 발생한 구조적 결측이다.

예를 들어,
- `transaction` 이벤트에서 `offer_id`가 없는 것은 정상이다.
- `transaction`이 아닌 이벤트에서 `amount`가 없는 것도 정상이다.
- `offer completed`가 아닌 이벤트에서 `event_reward`가 없는 것도 정상이다.

따라서 결측은 일괄적으로 데이터 품질 문제로 해석하지 않고,\
이벤트 구조에 따른 정상 결측과 실제 미매칭 결측을 구분해 봐야 한다.

# 전환율 분석용 퍼널 테이블 생성

received를 기준으로 viewed, completed 반응을 연결해 분석용 퍼널 테이블을 만든다.

- `viewed`, `completed`는 반드시 해당 오퍼의 `duration`(유효기간) 안에서 발생한 경우만 유효 반응으로 인정한다.
- `completed`는 단순 존재 여부가 아니라, 유효한 `received` 또는 `viewed`와 시간 순서상 연결될 때만 인정한다.
- `transaction`은 특정 offer와 직접 연결되지 않으므로, 이 구간에서는 프로모션 직접 전환 지표로 사용하지 않는다.

이 단계에선 "오퍼를 받은 뒤 실제로 열람했는가 / 완료했는가"를 받은 시점 기준으로 재구성해 분석하는 과정이다.

## 2-1. 이벤트 분리

퍼널의 분모와 분자를 만들기 위해 `received`, `viewed`, `completed` 이벤트를 분리한다.

In [29]:
# ============================================================
# received / viewed / completed 이벤트 분리
# 각 이벤트를 별도 테이블로 나누고, 이후 퍼널 분석을 위해 시간 컬럼명을 이벤트별로 구분
# ============================================================

# received 분리
# 오퍼를 받은 시점 기준 테이블
received = (
    df2[df2['event'] == 'offer received'][
        [
            'customer_id',       # 고객 ID
            'offer_id',          # 오퍼 ID
            'time',              # 오퍼 수신 시점
            'offer_type',        # 오퍼 유형
            'offer_reward',      # 오퍼 보상 금액
            'difficulty',        # 오퍼 완료 조건 금액
            'duration',          # 오퍼 유효 기간
            'web',               # 웹 채널 포함 여부
            'email',             # 이메일 채널 포함 여부
            'mobile',            # 모바일 채널 포함 여부
            'social',            # 소셜 채널 포함 여부
            'has_profile',       # 고객 정보 보유 여부
            'gender',            # 고객 성별
            'age',               # 고객 나이
            'income',            # 고객 소득
            'became_member_on'   # 멤버십 가입일
        ]
    ]
    .copy()
    .rename(columns={'time': 'time_received'})
)


# viewed 분리
# 오퍼를 열람한 시점만 따로 추출
viewed = (
    df2[df2['event'] == 'offer viewed'][
        [
            'customer_id',   # 고객 ID
            'offer_id',      # 오퍼 ID
            'time'           # 오퍼 열람 시점
        ]
    ]
    .copy()
    .rename(columns={'time': 'time_viewed'})
)

# completed 분리
# 오퍼를 완료한 시점만 따로 추출
completed = (
    df2[df2['event'] == 'offer completed'][
        [
            'customer_id',   # 고객 ID
            'offer_id',      # 오퍼 ID
            'time'           # 오퍼 완료 시점
        ]
    ]
    .copy()
    .rename(columns={'time': 'time_completed'})
)

# 분리 결과 행 수 확인
print("received 행 수:", len(received))
print("viewed 행 수:", len(viewed))
print("completed 행 수:", len(completed))

received 행 수: 76277
viewed 행 수: 57725
completed 행 수: 33182


## 시간순 정렬과 반응 단위 정의

동일 고객이 동일 offer를 여러 번 받을 수 있으므로,\
`(customer_id + offer_id)` 조합만으로는 하나의 고유 반응 단위를 정의할 수 없다.

따라서 본 분석에서는 `received` 이벤트 1건을 하나의 반응 단위(instance처럼)로 보고,\
그 뒤에 발생한 `viewed`, `completed`를 시간 순서와 `duration` 조건에 맞게 연결한다.

### 왜 seq 방식 대신 최근 유효 received 매칭을 쓰는가?
처음에는 n번째 `received`와 n번째 `completed`를 연결하는 방식도 고려할 수 있다.\
하지만 같은 고객이 같은 오퍼를 이전 오퍼 유효기간이 끝나기 전에 또 받은 경우,\
단순 순번 매칭은 잘못된 연결을 만들 수 있다.

- 그래서 다음 기준을 사용한다.
    1. `completed`보다 먼저 발생한 `received`만 후보로 본다.
    2. 그중 `duration` 안에 있는 가장 최근의 `received`를 연결한다.
    3. 조건에 맞는 `received`가 없으면 해당 `completed`는 분석용 매칭에서 제외한다.


### 추가 시간 컬럼
- `time_completed_rc` : `received → completed`용 completed 시점
- `time_completed_vc` : `viewed → completed`용 completed 시점

### 전환 플래그
- `converted_rv` : `received → viewed`
- `converted_rc` : `received → completed`
- `converted_vc` : `viewed → completed`
- `converted_rvc` : `received → viewed → completed`

In [ ]:
# 정렬 및 기준키 생성

# 이벤트별 시간 순서 정렬
# 같은 고객이 같은 오퍼를 여러 번 받을 수 있으므로 시간 기준 정렬이 필요
received = received.sort_values(
    ["customer_id", "offer_id", "time_received"]
).reset_index(drop=True)

viewed = viewed.sort_values(
    ["customer_id", "offer_id", "time_viewed"]
).reset_index(drop=True)

completed = completed.sort_values(
    ["customer_id", "offer_id", "time_completed"]
).reset_index(drop=True)

# received 기준 고유 행 번호 생성
# 이후 viewed / completed를 received와 1대1로 매칭할 때 기준키로 사용
received["received_idx"] = range(len(received))

# 오퍼 유효 종료 시점 계산
# duration은 일(day) 단위이므로 24를 곱해 시간(hour) 단위로 변환
received["offer_end_time"] = received["time_received"] + received["duration"] * 24

In [34]:
# ============================================================
# 공통 매칭 함수
# - event를 가장 최근의 유효한 base 이벤트에 연결
# - base가 event보다 먼저 발생한 경우만 인정
# - offer 유효기간(duration) 안의 이벤트만 인정
# - 1대1 매칭을 유지해 같은 event가 여러 base에 중복 연결되지 않도록 처리
# ============================================================
def match_event_to_base(
        base_df,           # 기준이 되는 이벤트 테이블(received 또는 viewed)
        event_df,          # base에 연결할 이벤트 테이블(viewed 또는 completed)
        base_time_col,     # base 이벤트의 시간 컬럼명
        event_time_col,    # 연결할 event의 시간 컬럼명
        matched_time_col,  # 매칭 후 result에 새로 붙일 시간 컬럼명
        base_id_col="received_idx"  # base 행을 구분하는 고유 식별 컬럼명
        ):


    # ============================================================
    # 0) 입력 검증
    # ============================================================
    required_base_cols = {"customer_id", "offer_id", base_id_col, base_time_col}
    required_event_cols = {"customer_id", "offer_id", event_time_col}

    # offer_end_time은 선택 컬럼으로 처리
    has_offer_end = "offer_end_time" in base_df.columns

    missing_base = required_base_cols - set(base_df.columns)
    missing_event = required_event_cols - set(event_df.columns)

    if missing_base:
        raise ValueError(f"base_df에 필요한 컬럼이 없습니다: {missing_base}")
    if missing_event:
        raise ValueError(f"event_df에 필요한 컬럼이 없습니다: {missing_event}")

    # base_id_col은 반드시 유일해야 이후 merge 시 중복 문제가 없음
    if base_df[base_id_col].duplicated().any():
        raise ValueError(
            f"{base_id_col} 컬럼에 중복이 있습니다. "
            "기준 테이블의 고유 식별자가 유일해야 합니다."
        )

    # ============================================================
    # 1) 필요한 컬럼만 추출 + 숫자형 시간 정리
    # ============================================================
    base_keep_cols = ["customer_id", "offer_id", base_id_col, base_time_col]
    if has_offer_end:
        base_keep_cols.append("offer_end_time")

    base_sorted = base_df[base_keep_cols].copy()
    event_sorted = event_df[["customer_id", "offer_id", event_time_col]].copy()

    # 시간 컬럼 숫자형 변환
    base_sorted[base_time_col] = pd.to_numeric(base_sorted[base_time_col], errors="coerce")
    event_sorted[event_time_col] = pd.to_numeric(event_sorted[event_time_col], errors="coerce")

    if has_offer_end:
        base_sorted["offer_end_time"] = pd.to_numeric(base_sorted["offer_end_time"], errors="coerce")

    # 핵심 키/시간 결측 제거
    base_sorted = base_sorted.dropna(subset=["customer_id", "offer_id", base_id_col, base_time_col])
    event_sorted = event_sorted.dropna(subset=["customer_id", "offer_id", event_time_col])

    # 정렬
    base_sorted = (
        base_sorted
        .sort_values(["customer_id", "offer_id", base_time_col])
        .reset_index(drop=True)
    )

    event_sorted = (
        event_sorted
        .sort_values(["customer_id", "offer_id", event_time_col])
        .reset_index(drop=True)
    )

    # ============================================================
    # 2) 그룹별 base 저장
    # ============================================================
    if has_offer_end:
        base_groups = {
            key: group[[base_id_col, base_time_col, "offer_end_time"]].to_numpy()
            for key, group in base_sorted.groupby(["customer_id", "offer_id"], sort=False)
        }
    else:
        base_groups = {
            key: group[[base_id_col, base_time_col]].to_numpy()
            for key, group in base_sorted.groupby(["customer_id", "offer_id"], sort=False)
        }

    # 매칭 결과 저장
    match_rows = []

    # ============================================================
    # 3) event별로 가장 최근의 유효한 base 탐색
    # ============================================================
    for key, event_group in event_sorted.groupby(["customer_id", "offer_id"], sort=False):

        bases = base_groups.get(key)

        # 해당 고객+오퍼의 base가 없으면 skip
        if bases is None or len(bases) == 0:
            continue

        base_ids = bases[:, 0]
        base_times = bases[:, 1].astype(float)

        if has_offer_end:
            offer_end_times = bases[:, 2].astype(float)
        else:
            # 종료 제한이 없으면 무한대로 처리
            offer_end_times = np.full(len(bases), np.inf)

        # 같은 방식에서 base는 1번만 사용
        used = np.zeros(len(bases), dtype=bool)

        for event_time in event_group[event_time_col].to_numpy(dtype=float):

            if pd.isna(event_time):
                continue

            # event_time 이전(또는 같은 시점)의 가장 최근 base 위치
            j = np.searchsorted(base_times, event_time, side="right") - 1

            while j >= 0:
                if (
                    (not used[j]) and
                    (base_times[j] <= event_time) and
                    (event_time <= offer_end_times[j])
                ):
                    match_rows.append({
                        base_id_col: base_ids[j],
                        matched_time_col: event_time
                    })

                    used[j] = True
                    break

                j -= 1

    # ============================================================
    # 4) 결과 병합
    # ============================================================
    result = base_df.copy()

    if not match_rows:
        result[matched_time_col] = pd.NA
        return result

    match_df = pd.DataFrame(match_rows)

    # 혹시 예외적으로 중복 매칭이 생기면 첫 번째만 유지
    match_df = match_df.drop_duplicates(subset=[base_id_col], keep="first")

    result = result.merge(
        match_df,
        on=base_id_col,
        how="left"
    )

    return result

## 4개 전환 기준 테이블 생성

In [35]:
# 1) received -> viewed 매칭
rv = match_event_to_base(
    base_df=received,                 # 기준: 받은 오퍼(received)
    event_df=viewed,                  # 연결 대상: 열람 이벤트(viewed)
    base_time_col="time_received",    # 기준 시간: 오퍼 받은 시점
    event_time_col="time_viewed",     # 연결할 이벤트 시간: 오퍼 본 시점
    matched_time_col="time_viewed"    # 결과로 붙일 컬럼명
)

In [ ]:
# 2) received -> completed 매칭
rc_match = match_event_to_base(
    base_df=received,                    # 기준: 받은 오퍼(received)
    event_df=completed,                  # 연결 대상: 완료 이벤트(completed)
    base_time_col="time_received",       # 기준 시간: 오퍼 받은 시점
    event_time_col="time_completed",     # 연결할 이벤트 시간: 오퍼 완료 시점
    matched_time_col="time_completed_rc" # 결과로 붙일 컬럼명(received→completed용)
)

In [ ]:
# ============================================================
# 3) viewed -> completed 매칭
# viewed가 붙은 received만 대상으로, viewed 이후 completed를 다시 매칭
# ============================================================
view_base = rv[rv["time_viewed"].notna()].copy()

vc_match = match_event_to_base(
    base_df=view_base,                  # 기준: viewed가 확인된 received
    event_df=completed,                 # 연결 대상: 완료 이벤트(completed)
    base_time_col="time_viewed",        # 기준 시간: 오퍼 본 시점
    event_time_col="time_completed",    # 연결할 이벤트 시간: 오퍼 완료 시점
    matched_time_col="time_completed_vc" # 결과로 붙일 컬럼명(viewed→completed용)
)

In [ ]:
# ============================================================
# 최종 received 기준 퍼널 테이블 생성
# ============================================================
funnel = received.copy()

# received -> viewed 매칭 결과 결합
funnel = funnel.merge(
    rv[["received_idx", "time_viewed"]],
    on="received_idx",
    how="left"
)

# received -> completed 매칭 결과 결합
funnel = funnel.merge(
    rc_match[["received_idx", "time_completed_rc"]],
    on="received_idx",
    how="left"
)


# viewed -> completed 매칭 결과 결합
# viewed 이후 completed까지 연결된 시점을 received 기준 테이블에 함께 붙임
funnel = funnel.merge(
    vc_match[["received_idx", "time_completed_vc"]],
    on="received_idx",
    how="left"
)

# 단계별 시간 차이 계산
funnel["rv_time_diff"] = funnel["time_viewed"] - funnel["time_received"]
funnel["rc_time_diff"] = funnel["time_completed_rc"] - funnel["time_received"]
funnel["vc_time_diff"] = funnel["time_completed_vc"] - funnel["time_viewed"]

# 이벤트 존재 여부 플래그 생성
# 시간값이 존재하면 1, 없으면 0으로 변환
funnel["has_viewed"] = funnel["time_viewed"].notna().astype(int)
funnel["has_completed_rc"] = funnel["time_completed_rc"].notna().astype(int)
funnel["has_completed_after_view"] = funnel["time_completed_vc"].notna().astype(int)

# 전환 플래그
funnel["converted_rv"] = funnel["has_viewed"]                   # received -> viewed 전환 여부
funnel["converted_rc"] = funnel["has_completed_rc"]             # received -> completed 전환 여부
funnel["converted_vc"] = funnel["has_completed_after_view"]     # viewed -> completed 전환 여부
funnel["converted_rvc"] = (
    (funnel["has_viewed"] == 1) &
    (funnel["has_completed_after_view"] == 1)
).astype(int)                                                   # received -> viewed -> completed 전환 여부


# 기존 코드 호환용 최종 전환 컬럼 생성
# 기존 converted_final 의미를 유지하기 위해 received -> completed 기준 사용
funnel["converted_final"] = funnel["converted_rc"]

# 상태 분류 파생 컬럼 생성
# 퍼널 진행 상태를 문자열 라벨로 구분
funnel["status"] = np.select(
    [
        funnel["has_completed_rc"] == 1,
        funnel["has_viewed"] == 0,
        (funnel["has_viewed"] == 1) & (funnel["has_completed_rc"] == 0)
    ],
    [
        "converted",
        "not_viewed",
        "viewed_not_converted"
    ],
    default="other"
)

### 퍼널 테이블 생성 결과 해석
여기서 만들어진 `funnel`은 모든 분석의 출발점이 되는 중간 테이블이다.

- 기준 단위는 `received` 1건이다.
- 각 received에 대해 유효한 `viewed`, `completed`를 연결한다.
- 아직은 중간 테이블이므로, 이후 채널/세그먼트/가입기간 같은 파생 컬럼을 추가해 최종 저장용 테이블로 확장한다.

In [ ]:
# ============================================================
# 채널 파생컬럼 생성
# - channel_count : 발송 채널 수
# - channel_combo : 발송 채널 조합
# ============================================================

# ============================================================
# channel_count
# 채널 관련 컬럼 목록 정의
channel_cols = ['web', 'email', 'mobile', 'social']

# 채널 컬럼 정리
# 결측치는 0으로 채우고, 채널 여부가 0/1 형태가 되도록 정수형으로 변환
funnel[channel_cols] = funnel[channel_cols].fillna(0).astype(int)

# 채널 수 파생 컬럼 생성
# 한 오퍼가 몇 개의 채널로 발송되었는지 계산
funnel['channel_count'] = funnel[channel_cols].sum(axis=1)
# ============================================================

# ============================================================
# channel_combo
# 채널 조합 파생 컬럼 생성
# 값이 1인 채널명만 이어 붙여 문자열로 저장
# 예: 'email+mobile', 'web+email+mobile'
funnel['channel_combo'] = ''

for col in channel_cols:
    funnel['channel_combo'] += np.where(
        funnel[col] == 1,
        col + '+',
        ''
    )

# 마지막 '+' 제거
funnel['channel_combo'] = funnel['channel_combo'].str.rstrip('+')

# 모든 채널 값이 0인 경우 'none'으로 표시
funnel.loc[funnel['channel_combo'] == '', 'channel_combo'] = 'none'
# ============================================================


print("channel_count / channel_combo 생성 완료")
display(
    funnel[
        ['web', 'email', 'mobile', 'social', 'channel_count', 'channel_combo']
    ].drop_duplicates().sort_values(['channel_count', 'channel_combo'])
)

channel_count / channel_combo 생성 완료


,web,email,mobile,social,channel_count,channel_combo
7,1,1,0,0,2,web+email
2,0,1,1,1,3,email+mobile+social
0,1,1,1,0,3,web+email+mobile
3,1,1,1,1,4,web+email+mobile+social


In [ ]:
# ============================================================
# 고객 세그먼트 파생컬럼 생성
# ============================================================

# 1) 성별 그룹
# 결측치는 별도 집단으로 남겨 세그먼트 분석 시 누락되지 않도록 한다.
funnel['gender_group'] = funnel['gender'].fillna('Missing')

# 2) 가입일 날짜형 변환: datetime 변환 후 year 추출
# 이후 가입 연도, 코호트, 가입기간 파생에 활용하기 위해 datetime으로 변환
became_member_str = funnel['became_member_on'].astype(str).str.replace('-', '', regex=False)

funnel['became_member_on_dt'] = pd.to_datetime(
    became_member_str,
    format='%Y%m%d',
    errors='coerce'
)

# 3) 가입 연도 = member_year
funnel['member_year'] = funnel['became_member_on_dt'].dt.year.astype('Int64')

# 4) 소득 구간
# 고객 소득을 해석하기 쉬운 범주형 구간으로 변환한다.
funnel['income_group'] = pd.cut(
    funnel['income'],
    bins=[0, 40000, 60000, 80000, 100000, 200000],
    labels=['~4만', '4~6만', '6~8만', '8~10만', '10만+']
).astype('object').fillna('Missing')

# 5) 나이 구간
# 연속형 나이를 대시보드와 비교 분석에 적합한 구간형 변수로 변환한다.
funnel['age_group'] = pd.cut(
    funnel['age'],
    bins=[0, 30, 40, 50, 60, 120],
    labels=['~30대', '40대', '50대', '60대', '70대+']
).astype('object').fillna('Missing')

# 6) 가입 코호트
# 가입 연도 기준으로 고객군을 묶어 코호트 비교가 가능하도록 한다.
funnel['member_cohort'] = pd.cut(
    funnel['member_year'],
    bins=[2013, 2015, 2017, 2019],
    right=False,
    labels=['2013-2014', '2015-2016', '2017-2018']
).astype('object').fillna('Missing')

print("세그먼트 파생컬럼 생성 완료")
display(
    funnel[
        [
            'gender', 'gender_group',
            'age', 'age_group',
            'income', 'income_group',
            'became_member_on', 'became_member_on_dt', 'member_year', 'member_cohort'
        ]
    ].head()
)

세그먼트 파생컬럼 생성 완료


,gender,gender_group,age,age_group,income,income_group,became_member_on,became_member_on_dt,member_year,member_cohort
0,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
1,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
2,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
3,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
4,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018


In [ ]:
# ============================================================
# 가입 경과일 / 오퍼 수신 횟수 파생컬럼 생성: 조준익
# ============================================================

# 1) 가입 경과일 계산 기준일
# 데이터의 마지막 관측 시점에 맞춰 가입 후 얼마나 경과했는지 계산한다.
last_date = pd.Timestamp('2018-08-01')

# 2) 가입 경과일
funnel['days_as_member'] = (
    last_date - funnel['became_member_on_dt']
).dt.days

# 3) 가입 경과일 구간화
# 가입기간에 따른 반응 차이를 보기 쉽도록 범주형 변수로 변환한다.
funnel['days_as_member_group'] = pd.cut(
    funnel['days_as_member'],
    bins=[0, 200, 500, 1000, 2000],
    labels=['~200일', '~500일', '~1000일', '1000일+']
).astype('object').fillna('Missing')

# 4) 고객별 오퍼 수신 횟수
# 동일 고객이 전체 분석 기간 동안 몇 번의 오퍼를 받았는지 계산한다.
offer_count_df = (
    funnel.groupby('customer_id')['offer_id']
    .count()
    .reset_index(name='offer_count')
)

funnel = funnel.merge(
    offer_count_df,
    on='customer_id',
    how='left'
)

display(
    funnel[
        [
            'customer_id',                # 고객 ID
            'became_member_on_dt',        # 멤버십 가입일
            'days_as_member',             # 가입 경과일
            'days_as_member_group',       # 가입 경과일 구간
            'offer_count'                 # 고객별 오퍼 수신 횟수
        ]
    ].head()
)

,customer_id,became_member_on_dt,days_as_member,days_as_member_group,offer_count
0,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5
1,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5
2,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5
3,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5
4,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5


In [ ]:
# ============================================================
# 고객 기준 전환 세그먼트 + 오퍼 전후 구매 비교 테이블 생성
# - 고객별 최종 전환 여부를 기준으로 segment 생성
# - transaction 이벤트를 분리해 전환 오퍼 수신 전후 7일 구매를 비교
# - 고객별 구매 횟수 / 구매 금액 요약 테이블 생성
# ============================================================


# 1) 고객 기준 segment 생성
# 고객이 한 번이라도 전환(converted_final=1)한 경우 'converted'
# 한 번도 전환하지 안한경우 'not_converted'
customer_segment = (
    funnel.groupby('customer_id', as_index=False)['converted_final']
    .max()
    .rename(columns={'converted_final': 'converted'})
)

# 전환 여부를 문자열 segment로 변환
customer_segment['segment'] = customer_segment['converted'].map({
    1: 'converted',
    0: 'not_converted'
})

# 2) transaction 이벤트만 분리
# 실제 구매 이벤트만 추출하여 비교용 테이블 생성
transactions = (
    df2[df2['event'] == 'transaction'][['customer_id', 'time', 'amount']]
    .copy()
    .rename(columns={'time': 'time_transaction'})
)

# amount 숫자형 변환
transactions['amount'] = pd.to_numeric(
    transactions['amount'],
    errors='coerce'
).fillna(0)     # 숫자로 변환되지 않는 값은 NaN 처리 후 0으로 대체

print("transaction 행 수:", len(transactions))
display(transactions.head())


# 3) 비교 기준 테이블 생성
# 전환된 오퍼(received 기준)를 잡고, 해당 시점 전후의 거래를 비교한다.
compare_base = funnel.loc[
    funnel['converted_final'] == 1,
    ['customer_id', 'time_received']
].copy()

print("전환된 offer 기준 compare_base 행 수:", len(compare_base))
display(compare_base.head())


# 4) 고객 기준 transaction 결합 및 전후 7일 플래그 생성
compare_merged = compare_base.merge(
    transactions,
    on='customer_id',
    how='left'
)       # received 시점을 기준으로 구매가 전 7일 / 후 7일 안에 발생했는지 구분

# 비교 기간 설정: 7일 = 168시간
window_hours = 7 * 24

# 오퍼 수신 전 7일 이내 구매 여부
compare_merged['before_flag'] = (
    (compare_merged['time_transaction'] >= compare_merged['time_received'] - window_hours) &
    (compare_merged['time_transaction'] < compare_merged['time_received'])
)

# 오퍼 수신 후 7일 이내 구매 여부
compare_merged['after_flag'] = (
    (compare_merged['time_transaction'] >= compare_merged['time_received']) &
    (compare_merged['time_transaction'] <= compare_merged['time_received'] + window_hours)
)

display(compare_merged.head())


# 5) 고객 기준 구매 횟수 / 구매 금액 집계
before_df = compare_merged[compare_merged['before_flag']].copy()
after_df = compare_merged[compare_merged['after_flag']].copy()

before_agg = (
    before_df.groupby('customer_id', as_index=False)
    .agg(
        before_count=('amount', 'count'),   # 전 7일 구매 횟수
        before_amount=('amount', 'sum')     # 전 7일 구매 금액 합계
    )
)

after_agg = (
    after_df.groupby('customer_id', as_index=False)
    .agg(
        after_count=('amount', 'count'),    # 후 7일 구매 횟수
        after_amount=('amount', 'sum')      # 후 7일 구매 금액 합계
    )
)

# 6) 고객 segment + 전/후 구매 집계 결과를 하나의 테이블로 결합
customer_compare = customer_segment.merge(
    before_agg,
    on='customer_id',
    how='left'
).merge(
    after_agg,
    on='customer_id',
    how='left'
)

# 결측값 처리
# 구매 이력이 없는 고객은 0으로 대체
for col in ['before_count', 'before_amount', 'after_count', 'after_amount']:
    customer_compare[col] = customer_compare[col].fillna(0)

# 구매 횟수 컬럼은 정수형으로 변환
customer_compare['before_count'] = customer_compare['before_count'].astype(int)
customer_compare['after_count'] = customer_compare['after_count'].astype(int)

print("customer_compare 생성 완료")
print("행 수:", len(customer_compare))
display(customer_compare.head())

transaction 행 수: 138953


,customer_id,time_transaction,amount
12654,02c083884c7d45b39cc68e1314fec56c,0,0.83
12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,0,34.56
12659,54890f68699049c2a04d415abc25e717,0,13.23
12670,b2f1cd155b864803ad8334cdf13c4bd2,0,19.51
12671,fe97aa22dd3e48c8b143116a8403dd52,0,18.97


전환된 offer 기준 compare_base 행 수: 33152


,customer_id,time_received
0,0009655768c64bdeb2e877511632db8f,576
3,0009655768c64bdeb2e877511632db8f,408
4,0009655768c64bdeb2e877511632db8f,504
7,0011e0d4e6b944f998e987f904e8c1e5,408
8,0011e0d4e6b944f998e987f904e8c1e5,168


,customer_id,time_received,time_transaction,amount,before_flag,after_flag
0,0009655768c64bdeb2e877511632db8f,576,228,22.16,False,False
1,0009655768c64bdeb2e877511632db8f,576,414,8.57,True,False
2,0009655768c64bdeb2e877511632db8f,576,528,14.11,True,False
3,0009655768c64bdeb2e877511632db8f,576,552,13.56,True,False
4,0009655768c64bdeb2e877511632db8f,576,576,10.27,False,True


customer_compare 생성 완료
행 수: 16994


,customer_id,converted,segment,before_count,before_amount,after_count,after_amount
0,0009655768c64bdeb2e877511632db8f,1,converted,4,44.81,12,166.01
1,00116118485d4dfda04fdbaba9a87b5c,0,not_converted,0,0.00,0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,1,converted,2,25.42,5,88.02
3,0020c2b971eb4e9188eac86d93036a77,1,converted,0,0.00,6,149.43
4,0020ccbbb6d84e358d3414a3ff76cffd,1,converted,10,136.26,11,137.78


# 최종

## 3-1. 메인 퍼널 테이블 `funnel_final`

In [ ]:
# ============================================================
# 메인 funnel 최종 저장용 컬럼 정리
# ============================================================
funnel_final = funnel[
    [
        # 원본 식별 / 오퍼 속성
        'customer_id', 'offer_id', 'time_received',
        'offer_type', 'offer_reward', 'difficulty', 'duration',

        # 채널 원본
        'web', 'email', 'mobile', 'social',

        # 퍼널 시간
        'offer_end_time', 'time_viewed', 'time_completed_rc', 'time_completed_vc',

        # 시간차
        'rv_time_diff', 'rc_time_diff', 'vc_time_diff',

        # 퍼널 플래그
        'has_viewed', 'has_completed_rc', 'has_completed_after_view',

        # 전환 플래그
        'converted_rv', 'converted_rc', 'converted_vc', 'converted_rvc', 'converted_final',

        # 상태
        'status',

        # 채널 파생
        'channel_count', 'channel_combo',

        # 세그먼트 파생
        'has_profile', 'gender_group', 'age_group', 
        'income_group', 'member_year', 'member_cohort',

        # 확장 파생
        'days_as_member', 'days_as_member_group', 'offer_count'
        
    ]
].copy()

print("funnel_final 생성 완료")
print("행 수:", len(funnel_final))
print("열 수:", funnel_final.shape[1])
display(funnel_final.head())

funnel_final 생성 완료
행 수: 76277
열 수: 38


,customer_id,offer_id,time_received,offer_type,offer_reward,difficulty,duration,web,email,mobile,social,offer_end_time,time_viewed,time_completed_rc,time_completed_vc,rv_time_diff,rc_time_diff,vc_time_diff,has_viewed,has_completed_rc,has_completed_after_view,converted_rv,converted_rc,converted_vc,converted_rvc,converted_final,status,channel_count,channel_combo,has_profile,gender_group,age_group,income_group,member_year,member_cohort,days_as_member,days_as_member_group,offer_count
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,discount,2.0,10.0,7.0,1,1,1,0,744.0,NaN,576.0,NaN,NaN,0.0,NaN,0,1,0,0,1,0,0,1,converted,3,web+email+mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,336,informational,0.0,0.0,4.0,1,1,1,0,432.0,372.0,NaN,NaN,36.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,3,web+email+mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,168,informational,0.0,0.0,3.0,0,1,1,1,240.0,192.0,NaN,NaN,24.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,3,email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,408,bogo,5.0,5.0,5.0,1,1,1,1,528.0,456.0,414.0,NaN,48.0,6.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,504,discount,2.0,10.0,10.0,1,1,1,1,744.0,540.0,528.0,NaN,36.0,24.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5


`funnel_final`은 가장 범용성이 높은 기본 테이블이다.

- 전체 오퍼를 포함하므로 열람률, 세그먼트별 반응, 채널 반응 분석에 적합하다.
- 반면 `informational` 오퍼까지 포함하므로 완료율 비교를 바로 수행하는 테이블로 쓰기에는 주의가 필요하다.
- 따라서 완료/최종전환 비교는 다음 단계의 `funnel_offer_final`을 사용하는 것이 적절하다.

In [ ]:
# ============================================================
# B. 고객 기준 최종 저장용 컬럼만 정리
# ============================================================
customer_compare_final = customer_compare[
    [
        'customer_id',
        'segment',
        'before_count',
        'before_amount',
        'after_count',
        'after_amount'
    ]
].copy()

print("customer_compare_final 생성 완료")
display(customer_compare_final.head())

customer_compare_final 생성 완료


,customer_id,segment,before_count,before_amount,after_count,after_amount
0,0009655768c64bdeb2e877511632db8f,converted,4,44.81,12,166.01
1,00116118485d4dfda04fdbaba9a87b5c,not_converted,0,0.00,0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,converted,2,25.42,5,88.02
3,0020c2b971eb4e9188eac86d93036a77,converted,0,0.00,6,149.43
4,0020ccbbb6d84e358d3414a3ff76cffd,converted,10,136.26,11,137.78


## 3-2. 대시보드용 보조 테이블 `funnel_offer_final`

In [ ]:
# ============================================================
# 완료/최종전환 분석용 보조 테이블
# informational 제외
# ompletion 관련 분석은 이 테이블 기준으로 사용
# ============================================================

funnel_offer_final = funnel_final[
    funnel_final['offer_type'] != 'informational'
].copy()

print("funnel_offer_final 생성 완료")
print("행 수:", len(funnel_offer_final))
print("열 수:", funnel_offer_final.shape[1])

print("\n[offer_type 분포]")
print(funnel_offer_final['offer_type'].value_counts(dropna=False))

print("\n[전환 플래그 분포]")
flag_cols = ['converted_rc', 'converted_vc', 'converted_rvc', 'converted_final']
for col in flag_cols:
    print(f"\n{col}")
    print(funnel_offer_final[col].value_counts(dropna=False).sort_index())

display(funnel_offer_final.head())

funnel_offer_final 생성 완료
행 수: 61042
열 수: 38

[offer_type 분포]
offer_type
discount    30543
bogo        30499
Name: count, dtype: int64

[전환 플래그 분포]

converted_rc
converted_rc
0    27890
1    33152
Name: count, dtype: int64

converted_vc
converted_vc
0    37546
1    23496
Name: count, dtype: int64

converted_rvc
converted_rvc
0    37546
1    23496
Name: count, dtype: int64

converted_final
converted_final
0    27890
1    33152
Name: count, dtype: int64


,customer_id,offer_id,time_received,offer_type,offer_reward,difficulty,duration,web,email,mobile,social,offer_end_time,time_viewed,time_completed_rc,time_completed_vc,rv_time_diff,rc_time_diff,vc_time_diff,has_viewed,has_completed_rc,has_completed_after_view,converted_rv,converted_rc,converted_vc,converted_rvc,converted_final,status,channel_count,channel_combo,has_profile,gender_group,age_group,income_group,member_year,member_cohort,days_as_member,days_as_member_group,offer_count
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,discount,2.0,10.0,7.0,1,1,1,0,744.0,NaN,576.0,NaN,NaN,0.0,NaN,0,1,0,0,1,0,0,1,converted,3,web+email+mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,408,bogo,5.0,5.0,5.0,1,1,1,1,528.0,456.0,414.0,NaN,48.0,6.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,504,discount,2.0,10.0,10.0,1,1,1,1,744.0,540.0,528.0,NaN,36.0,24.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,168,bogo,5.0,5.0,5.0,1,1,1,1,288.0,216.0,NaN,NaN,48.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,4,web+email+mobile+social,0,Missing,Missing,Missing,<NA>,Missing,NaN,Missing,2
6,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,576,bogo,5.0,5.0,5.0,1,1,1,1,696.0,630.0,NaN,NaN,54.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,4,web+email+mobile+social,0,Missing,Missing,Missing,<NA>,Missing,NaN,Missing,2


### 테이블별 사용 기준

- `funnel_final`
    - 전체 오퍼를 포함한 기본 퍼널 테이블
    - received 수, viewed 수, 열람률, 채널 반응, 세그먼트별 열람 반응 분석에 사용
    - `informational` 오퍼도 포함하므로 전체 오퍼 분포를 볼 때 적합하다.

- `funnel_offer_final`
    - `informational`을 제외한 완료/최종전환 분석용 테이블
    - completed 수, 완료율, 최종 전환율, reward / difficulty / duration 비교에 사용
    - `bogo`, `discount`처럼 완료 조건이 있는 오퍼 비교에 적합하다.

- `customer_compare_final`
    - 고객 기준 전환 여부와 오퍼 전후 구매 비교 테이블
    - 전환 고객 / 비전환 고객 비교, 전환 전후 구매 횟수·금액 비교에 사용

In [ ]:
## 4. 저장 전 검수 및 CSV 저장

In [ ]:
check_basic_info(funnel_final, 'funnel_final')


funnel_final의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,76277
1,열 개수,38
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
time_completed_vc,float64,23496,30.80,52781,69.20,120
vc_time_diff,float64,23496,30.80,52781,69.20,41
time_completed_rc,float64,33152,43.46,43125,56.54,120
rc_time_diff,float64,33152,43.46,43125,56.54,41
time_viewed,float64,56567,74.16,19710,25.84,120
rv_time_diff,float64,56567,74.16,19710,25.84,41
days_as_member,float64,66501,87.18,9776,12.82,1707
member_year,Int64,66501,87.18,9776,12.82,6
customer_id,str,76277,100.00,0,0.00,16994
offer_end_time,float64,76277,100.00,0,0.00,22


[테이블 요약]


,customer_id,offer_id,time_received,offer_type,offer_reward,difficulty,duration,web,email,mobile,social,offer_end_time,time_viewed,time_completed_rc,time_completed_vc,rv_time_diff,rc_time_diff,vc_time_diff,has_viewed,has_completed_rc,has_completed_after_view,converted_rv,converted_rc,converted_vc,converted_rvc,converted_final,status,channel_count,channel_combo,has_profile,gender_group,age_group,income_group,member_year,member_cohort,days_as_member,days_as_member_group,offer_count
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,discount,2.0,10.0,7.0,1,1,1,0,744.0,NaN,576.0,NaN,NaN,0.0,NaN,0,1,0,0,1,0,0,1,converted,3,web+email+mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,336,informational,0.0,0.0,4.0,1,1,1,0,432.0,372.0,NaN,NaN,36.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,3,web+email+mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,168,informational,0.0,0.0,3.0,0,1,1,1,240.0,192.0,NaN,NaN,24.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,3,email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,408,bogo,5.0,5.0,5.0,1,1,1,1,528.0,456.0,414.0,NaN,48.0,6.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,504,discount,2.0,10.0,10.0,1,1,1,1,744.0,540.0,528.0,NaN,36.0,24.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5


In [ ]:
check_basic_info(funnel_offer_final, 'funnel_offer_final')


funnel_offer_final의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,61042
1,열 개수,38
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
time_completed_vc,float64,23496,38.49,37546,61.51,120
vc_time_diff,float64,23496,38.49,37546,61.51,41
time_completed_rc,float64,33152,54.31,27890,45.69,120
rc_time_diff,float64,33152,54.31,27890,45.69,41
time_viewed,float64,46620,76.37,14422,23.63,120
rv_time_diff,float64,46620,76.37,14422,23.63,41
days_as_member,float64,53201,87.15,7841,12.85,1705
member_year,Int64,53201,87.15,7841,12.85,6
customer_id,str,61042,100.00,0,0.00,16928
offer_end_time,float64,61042,100.00,0,0.00,16


[테이블 요약]


,customer_id,offer_id,time_received,offer_type,offer_reward,difficulty,duration,web,email,mobile,social,offer_end_time,time_viewed,time_completed_rc,time_completed_vc,rv_time_diff,rc_time_diff,vc_time_diff,has_viewed,has_completed_rc,has_completed_after_view,converted_rv,converted_rc,converted_vc,converted_rvc,converted_final,status,channel_count,channel_combo,has_profile,gender_group,age_group,income_group,member_year,member_cohort,days_as_member,days_as_member_group,offer_count
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,discount,2.0,10.0,7.0,1,1,1,0,744.0,NaN,576.0,NaN,NaN,0.0,NaN,0,1,0,0,1,0,0,1,converted,3,web+email+mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,408,bogo,5.0,5.0,5.0,1,1,1,1,528.0,456.0,414.0,NaN,48.0,6.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,504,discount,2.0,10.0,10.0,1,1,1,1,744.0,540.0,528.0,NaN,36.0,24.0,NaN,1,1,0,1,1,0,0,1,converted,4,web+email+mobile+social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,168,bogo,5.0,5.0,5.0,1,1,1,1,288.0,216.0,NaN,NaN,48.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,4,web+email+mobile+social,0,Missing,Missing,Missing,<NA>,Missing,NaN,Missing,2
6,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,576,bogo,5.0,5.0,5.0,1,1,1,1,696.0,630.0,NaN,NaN,54.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,4,web+email+mobile+social,0,Missing,Missing,Missing,<NA>,Missing,NaN,Missing,2


In [ ]:
check_basic_info(customer_compare_final, 'customer_compare_final')


customer_compare_final의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,16994
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
customer_id,str,16994,100.0,0,0.0,16994
after_amount,float64,16994,100.0,0,0.0,10399
before_amount,float64,16994,100.0,0,0.0,8455
after_count,int64,16994,100.0,0,0.0,42
before_count,int64,16994,100.0,0,0.0,33
segment,str,16994,100.0,0,0.0,2


[테이블 요약]


,customer_id,segment,before_count,before_amount,after_count,after_amount
0,0009655768c64bdeb2e877511632db8f,converted,4,44.81,12,166.01
1,00116118485d4dfda04fdbaba9a87b5c,not_converted,0,0.00,0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,converted,2,25.42,5,88.02
3,0020c2b971eb4e9188eac86d93036a77,converted,0,0.00,6,149.43
4,0020ccbbb6d84e358d3414a3ff76cffd,converted,10,136.26,11,137.78


In [ ]:
# ============================================================
# 최종 CSV 저장
# ============================================================
funnel_final.to_csv('data/funnel_final.csv', index=False, encoding='utf-8-sig')
funnel_offer_final.to_csv('data/funnel_offer_final.csv', index=False, encoding='utf-8-sig')
customer_compare_final.to_csv('data/ustomer_compare_final.csv', index=False, encoding='utf-8-sig')

print("CSV 저장 완료")
print("- funnel_final.csv")
print("- funnel_offer_final.csv")
print("- customer_compare_final.csv")

CSV 저장 완료
- funnel_final.csv
- funnel_offer_final.csv
- customer_compare_final.csv


## 컬럼 설명

### `funnel_final` 주요 컬럼
| 컬럼명 | 의미 |
|---|---|
| customer_id | 고객 ID |
| offer_id | 오퍼 ID |
| time_received | 고객이 오퍼를 받은 시점 |
| offer_type | 오퍼 유형 |
| offer_reward | 오퍼 보상 금액 |
| difficulty | 오퍼 완료 조건 금액 |
| duration | 오퍼 유효 기간(일 단위) |
| web | 웹 채널 포함 여부 |
| email | 이메일 채널 포함 여부 |
| mobile | 모바일 채널 포함 여부 |
| social | 소셜 채널 포함 여부 |
| offer_end_time | 오퍼 종료 시점 (`time_received + duration*24`) |
| time_viewed | received에 연결된 viewed 시점 |
| time_completed_rc | received 기준 completed 시점 |
| time_completed_vc | viewed 이후 completed 시점 |
| rv_time_diff | received → viewed 시간차 |
| rc_time_diff | received → completed 시간차 |
| vc_time_diff | viewed → completed 시간차 |
| has_viewed | viewed 발생 여부 |
| has_completed_rc | received 이후 completed 여부 |
| has_completed_after_view | viewed 이후 completed 여부 |
| converted_rv | received 대비 viewed 전환 여부 |
| converted_rc | received 대비 completed 전환 여부 |
| converted_vc | viewed 대비 completed 전환 여부 |
| converted_rvc | received 이후 viewed도 했고 completed도 했는지 여부 |
| converted_final | 최종 대표 전환 여부 |
| status | 오퍼 상태 분류 (`converted`, `not_viewed`, `viewed_not_converted`) |
| channel_count | 발송 채널 수 |
| channel_combo | 발송 채널 조합 문자열 |
| has_profile | 프로필 정보 보유 여부 |
| gender_group | 성별 그룹 |
| age_group | 나이 구간 |
| income_group | 소득 구간 |
| member_year | 가입 연도 |
| member_cohort | 가입 코호트 구간 |
| days_as_member | 가입 후 경과일 |
| days_as_member_group | 가입 경과일 구간 |
| offer_count | 고객별 오퍼 수신 횟수 |

### `customer_compare_final` 주요 컬럼
| 컬럼명 | 의미 |
|---|---|
| customer_id | 고객 ID |
| segment | 고객 기준 전환 여부 (`converted`, `not_converted`) |
| before_count | 전환 오퍼 수신 전 7일 구매 횟수 |
| before_amount | 전환 오퍼 수신 전 7일 구매 금액 합계 |
| after_count | 전환 오퍼 수신 후 7일 구매 횟수 |
| after_amount | 전환 오퍼 수신 후 7일 구매 금액 합계 |